## Постановка задачи

Дана сеточная функция:
$$
\{ (x_i, y_i) \}_{i=0}^{n-1}, \quad a = x_0 < x_1 < \dots < x_{n-1} = b
$$
где $n$ — количество узлов интерполяции.

Требуется построить интерполяционный сплайн $S(x)$ с натяжением, удовлетворяющий условиям:

1. **Интерполяция:**
   $$
   S(x_i) = y_i, \quad i = 0,\dots,n-1
   $$

2. **Гладкость:**
   $$
   S(x) \in C^2[a,b]
   $$

3. **Граничные условия:** естественные сплайны
   $$
   S''(x_0) = S''(x_{n-1}) = 0
   $$

## Математическая модель

### 1. Весовые коэффициенты (модификация для натяжения)

Для каждого внутреннего узла $x_i$, $i=1,\dots,n-2$, вычисляются вспомогательные величины:

- Шаги сетки:
  $$
  h_i = x_{i+1} - x_i, \quad i=0,\dots,n-2
  $$

- Отношения наклонов:
  $$
  s_i = \frac{y_{i+1} - y_i}{h_i}, \quad i=0,\dots,n-2
  $$

- Вспомогательные параметры для $i=1,\dots,n-2$:
  $$
  L_i = \left( \frac{s_{i-1}}{s_i} - 2 \right) \frac{h_i}{h_{i-1}}
  $$
  $$
  U_i = 
  \begin{cases}
  \frac{1}{\left( \frac{s_i}{s_{i-1}} - 2 \right) \frac{h_{i-1}}{h_i}}, & \text{если знаменатель } > 0 \\
  +\infty, & \text{иначе}
  \end{cases}
  $$

- Промежуточный коэффициент:
  $$
  r_i = 
  \begin{cases}
  \max(1, L_i), & L_i > 0 \\
  1, & L_i \le 0
  \end{cases}
  $$
  $$
  r_i = \min(r_i, U_i)
  $$

- Весовые коэффициенты (рекуррентно):
  $$
  w_0 = 1, \quad w_1 = 1
  $$
  $$
  w_{i+1} = w_i \cdot r_i, \quad i=1,\dots,n-2
  $$

### 2. Система уравнений для моментов

Искомый сплайн на отрезке $[x_i, x_{i+1}]$ имеет вид:
$$
S_i(x) = \frac{M_i}{6h_i}(x_{i+1}-x)^3 + \frac{M_{i+1}}{6h_i}(x-x_i)^3 + 
\left(y_i - \frac{M_i h_i^2}{6}\right)\frac{x_{i+1}-x}{h_i} + 
\left(y_{i+1} - \frac{M_{i+1} h_i^2}{6}\right)\frac{x-x_i}{h_i}
$$
где $M_i = S''(x_i)$ — неизвестные моменты.

Система для определения $M_i$ ($i=0,\dots,n-1$):

$$
\begin{cases}
M_0 = 0 \\
\frac{w_i h_{i-1}}{6}M_{i-1} + \frac{w_i h_{i-1} + w_{i+1} h_i}{3}M_i + \frac{w_{i+1} h_i}{6}M_{i+1} = w_{i+1}\frac{y_{i+1}-y_i}{h_i} - w_i\frac{y_i-y_{i-1}}{h_{i-1}}, \\
\quad i=1,\dots,n-2 \\
M_{n-1} = 0
\end{cases}
$$

В матричной форме:
$$
\mathbf{C} \mathbf{M} = \mathbf{d}
$$
где для $i=1,\dots,n-2$:
$$
\begin{aligned}
C_{i,i-1} &= \frac{w_i h_{i-1}}{6} \\
C_{i,i} &= \frac{w_i h_{i-1} + w_{i+1} h_i}{3} \\
C_{i,i+1} &= \frac{w_{i+1} h_i}{6} \\
d_i &= w_{i+1}\frac{y_{i+1}-y_i}{h_i} - w_i\frac{y_i-y_{i-1}}{h_{i-1}}
\end{aligned}
$$
и $C_{0,0}=C_{n-1,n-1}=1$, $d_0=d_{n-1}=0$.

### 3. Частный случай — кубический сплайн

При $w_i \equiv 1$ получаем классический кубический сплайн:
$$
\frac{h_{i-1}}{6}M_{i-1} + \frac{h_{i-1}+h_i}{3}M_i + \frac{h_i}{6}M_{i+1} = \frac{y_{i+1}-y_i}{h_i} - \frac{y_i-y_{i-1}}{h_{i-1}}
$$

### 4. Вычисление значений сплайна

После решения системы и нахождения $M_i$, значение в произвольной точке $x \in [x_i, x_{i+1}]$:
$$
\begin{aligned}
S(x) &= \frac{M_i}{6h_i}(x_{i+1}-x)^3 + \frac{M_{i+1}}{6h_i}(x-x_i)^3 \\
&+ \left(y_i - \frac{M_i h_i^2}{6}\right)\frac{x_{i+1}-x}{h_i} + \left(y_{i+1} - \frac{M_{i+1} h_i^2}{6}\right)\frac{x-x_i}{h_i}
\end{aligned}
$$

## Алгоритм

1. **Вход:** узлы $(x_i, y_i)$, флаг вычисления весов
2. **Вычисление весов** (если требуется):
   - Вычислить $h_i$, $s_i$
   - Вычислить $r_i$ с ограничениями
   - Рекуррентно вычислить $w_i$
3. **Построение матрицы системы:**
   - Заполнить трёхдиагональную матрицу $\mathbf{C}$
   - Заполнить правую часть $\mathbf{d}$
4. **Решение системы:** $\mathbf{M} = \mathbf{C}^{-1}\mathbf{d}$
5. **Вычисление сплайна:**
   - Для каждого отрезка $[x_i, x_{i+1}]$
   - Вычислить значения по формуле (4) в равномерных точках отрезка
6. **Выход:** массивы координат сплайна

In [58]:
import numpy as np
import plotly.graph_objects as go

def f(x):
    # return np.exp(x) * x
    # return np.sign(x)\
    return np.exp(x)*np.sin(np.pi*x) + 2


def f_double_prime(x):
    return np.exp(x) * ( (1 - np.pi**2) * np.sin(np.pi * x) + 2 * np.pi * np.cos(np.pi * x) )

# Внутри функции splain:



# def compute_w(x, y):
#     h = np.diff(x)
#     s = np.diff(y) / h
#     w = [1, 1] 
#     for i in range(1, len(h)):
#         L = (s[i-1]/s[i] - 2) * (h[i]/h[i-1])
#         coef = (s[i]/s[i-1] - 2) * (h[i-1]/h[i])
#         U = np.inf if coef <= 0 else 1.0 / coef
#         ri = max(1.0, L) if L > 0 else 1.0
#         if ri > U:
#             ri = U
#         w.append(w[-1] * ri)
#     return np.array(w)


def compute_w(x, y):
    h = np.diff(x)
    s = np.diff(y) / h
    eps = 0

    w = [1,1] 
    for i in range(1, len(h)):
        if abs(s[i]) < eps:
            ri = 1.0
        else:
            L = (s[i-1]/(s[i] + eps) - 2) * (h[i]/h[i-1])
            coef = (s[i]/(s[i-1] + eps) - 2) * (h[i-1]/h[i])
            U = np.inf if coef <= 0 else 1.0 / (coef + eps)
            ri = max(1.0, L) if L > 0 else 1.0
            if ri > U:
                ri = U
        
        w.append(w[-1] * ri ) 
    
    return np.array(w) 



def splain(x_nodes,y_nodes,is_compute_weights=True,weights_input=None):
    n_points = len(x_nodes)  
    h_arr = np.diff(x_nodes) 

    if is_compute_weights:
        weights = compute_w(x_nodes, y_nodes)
    # else:
    #     weights = np.ones(len(x_nodes))

    if weights_input is not None:
        weights = weights_input
    # weights = [1.,  1. ,     1.,    1., 1.,   1.,  1.,   1.,   1.,    1.,      0.00001, 1.,      0.00001, 1.,      1.,      1.,      1.,      1.,    1.,      1.]
    weights[5] = 1
    

    C = np.zeros((n_points, n_points))
    d = np.zeros(n_points)

    C[0, 0] = 1
    C[n_points-1, n_points-1] = 1
    d[0] = 0
    d[n_points-1] = 0
    # d[0] = f_double_prime(x_nodes[0])
    # d[n_points-1] = f_double_prime(x_nodes[-1])

    for i in range(1, n_points-1):
        C[i, i-1] = weights[i] * h_arr[i-1] / 6
        C[i, i] = (weights[i] * h_arr[i-1] + weights[i+1] * h_arr[i]) / 3
        C[i, i+1] = weights[i+1] * h_arr[i] / 6
        d[i] = (weights[i+1] * (y_nodes[i+1] - y_nodes[i]) / h_arr[i] - 
                weights[i] * (y_nodes[i] - y_nodes[i-1]) / h_arr[i-1])

    M = np.linalg.solve(C, d)
    # M[0] = 1
    # M[-1] = 1

    n_spline_points = 100
    my_spline_x = []
    my_spline_y = []
    x_hist = []


    for i in range(1, n_points):
        x_start = x_nodes[i-1]
        x_end = x_nodes[i]
        h = h_arr[i-1]
        
        x_local = np.linspace(x_start, x_end, n_spline_points)
        
        for x in x_local:
            term1 = M[i-1] * ((x_end - x)**3) / (6 * h)
            term2 = M[i] * ((x - x_start)**3) / (6 * h)
            term3 = (y_nodes[i-1] - M[i-1] * h**2 / 6) * (x_end - x) / h
            term4 = (y_nodes[i] - M[i] * h**2 / 6) * (x - x_start) / h
            
            y_spline = term1 + term2 + term3 + term4
            my_spline_x.append(x)
            my_spline_y.append(y_spline)
            x_hist.append(x)

    return my_spline_x, my_spline_y, x_hist,weights


x_nodes = np.array([8.09, 8.19, 8.7, 9.2, 10, 12, 15, 20])
y_nodes = np.array([2.76429e-5, 4.37498e-2, 0.169183, 0.469428, 
                        0.94374, 0.998636, 0.999916, 0.999994])



# a = 0
# b = 1
# n_points = 5
# x_nodes = np.linspace(a, b, n_points)
# y_nodes = f(x_nodes)


# x_acc = np.linspace(a, b, 100)
# y_acc = f(x_acc)
splain_tension_x, splain_tension_y,_, w= splain(x_nodes, y_nodes,is_compute_weights=True)
# splain_base_x, splain_base_y,_ , w= splain(x_nodes, y_nodes,is_compute_weights=False)



fig = go.Figure()
fig.add_trace(go.Scatter(x=splain_tension_x, y=splain_tension_y,mode='lines',name='Сплайн c натяжением'))
fig.add_trace(go.Scatter(x=splain_base_x, y=splain_base_y,mode='lines',name='Обычный Сплайн'))
# fig.add_trace(go.Scatter(x=x_acc, y=y_acc,mode='lines',name='Заданная функция',marker=dict(size=8)))
fig.update_layout(title='Интерполяция сплайнами',xaxis_title='x',yaxis_title='y')
fig.show()



In [57]:
w

array([1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
       1.00000000e+00, 4.90012023e+01, 4.58145928e+03, 1.93569918e+05])

In [ ]:
a = 0
b = 1
n_points = 10
x_nodes = np.linspace(a, b, n_points)
y_nodes = f(x_nodes)


# x_acc = np.linspace(a, b,900 )
# y_acc = f(x_acc)
# splain_tension_x, splain_tension_y = splain(x_nodes, y_nodes,is_compute_weights=True)
splain_base_x, splain_base_y,x_hist = splain(x_nodes, y_nodes,is_compute_weights=False)\

print(len(x_hist))
# y_acc = f(splain_base_x)
y_acc = (np.exp(np.array(splain_base_x))*np.sin(np.pi*np.array(splain_base_x))+2)

fig = go.Figure()
# fig.add_trace(go.Scatter(x=splain_tension_x, y=splain_tension_y,mode='lines',name='Сплайн c натяжением'))
fig.add_trace(go.Scatter(x=splain_base_x, y=y_acc - splain_base_y,mode='lines',name='Обычный Сплайн'))
# fig.add_trace(go.Scatter(x=x_acc, y=y_acc,mode='lines',name='Заданная функция',marker=dict(size=8)))
fig.update_layout(title='Интерполяция сплайнами',xaxis_title='x',yaxis_title='y')
fig.show()


a = 0
b = 1
n_points = 20
x_nodes = np.linspace(a, b, n_points)
y_nodes = f(x_nodes)


# x_acc = np.linspace(a, b,900 )
# y_acc = f(x_acc)
# splain_tension_x, splain_tension_y = splain(x_nodes, y_nodes,is_compute_weights=True)
splain_base_x, splain_base_y,x_hist = splain(x_nodes, y_nodes,is_compute_weights=False)\

print(len(x_hist))
# y_acc = f(splain_base_x)
y_acc = (np.exp(np.array(splain_base_x))*np.sin(np.pi*np.array(splain_base_x))+2)

fig = go.Figure()
# fig.add_trace(go.Scatter(x=splain_tension_x, y=splain_tension_y,mode='lines',name='Сплайн c натяжением'))
fig.add_trace(go.Scatter(x=splain_base_x, y=y_acc - splain_base_y,mode='lines',name='Обычный Сплайн'))
# fig.add_trace(go.Scatter(x=x_acc, y=y_acc,mode='lines',name='Заданная функция',marker=dict(size=8)))
fig.update_layout(title='Интерполяция сплайнами',xaxis_title='x',yaxis_title='y')
fig.show()









100


100


In [33]:
a = 0
b = 1
n_points = 20
x_nodes = np.linspace(a, b, n_points)
y_nodes = f(x_nodes)


# x_acc = np.linspace(a, b,900 )
# y_acc = f(x_acc)
# splain_tension_x, splain_tension_y = splain(x_nodes, y_nodes,is_compute_weights=True)
splain_base_x, splain_base_y,x_hist = splain(x_nodes, y_nodes,is_compute_weights=False)\

print(len(x_hist))
# y_acc = f(splain_base_x)
y_acc = (np.exp(np.array(splain_base_x))*np.sin(np.pi*np.array(splain_base_x))+2)

fig = go.Figure()
# fig.add_trace(go.Scatter(x=splain_tension_x, y=splain_tension_y,mode='lines',name='Сплайн c натяжением'))
fig.add_trace(go.Scatter(x=splain_base_x, y=y_acc - splain_base_y,mode='lines',name='Обычный Сплайн'))
# fig.add_trace(go.Scatter(x=x_acc, y=y_acc,mode='lines',name='Заданная функция',marker=dict(size=8)))
fig.update_layout(title='Интерполяция сплайнами',xaxis_title='x',yaxis_title='y')
fig.show()





100


In [43]:
a = 0
b = 1
n_points = 10
x_nodes = np.linspace(a, b, n_points)
y_nodes = f(x_nodes)


# x_acc = np.linspace(a, b,900 )
# y_acc = f(x_acc)
# splain_tension_x, splain_tension_y = splain(x_nodes, y_nodes,is_compute_weights=True)
splain_base_x_1, splain_base_y_1,x_hist = splain(x_nodes, y_nodes,is_compute_weights=False)\

print(len(x_hist))
# y_acc = f(splain_base_x)
y_acc_1 = (np.exp(np.array(splain_base_x_1))*np.sin(np.pi*np.array(splain_base_x_1))+2)

# fig = go.Figure()
# # fig.add_trace(go.Scatter(x=splain_tension_x, y=splain_tension_y,mode='lines',name='Сплайн c натяжением'))
# fig.add_trace(go.Scatter(x=splain_base_x, y=y_acc - splain_base_y,mode='lines',name='Обычный Сплайн'))
# # fig.add_trace(go.Scatter(x=x_acc, y=y_acc,mode='lines',name='Заданная функция',marker=dict(size=8)))
# fig.update_layout(title='Интерполяция сплайнами',xaxis_title='x',yaxis_title='y')
# fig.show()


a = 0
b = 1
n_points = 20
x_nodes = np.linspace(a, b, n_points)
y_nodes = f(x_nodes)


# x_acc = np.linspace(a, b,900 )
# y_acc = f(x_acc)
# splain_tension_x, splain_tension_y = splain(x_nodes, y_nodes,is_compute_weights=True)
splain_base_x_2, splain_base_y_2,x_hist = splain(x_nodes, y_nodes,is_compute_weights=False)\

print(len(x_hist))
# y_acc = f(splain_base_x)
y_acc_2 = (np.exp(np.array(splain_base_x_2))*np.sin(np.pi*np.array(splain_base_x_2))+2)

fig = go.Figure()
# fig.add_trace(go.Scatter(x=splain_tension_x, y=splain_tension_y,mode='lines',name='Сплайн c натяжением'))
fig.add_trace(go.Scatter(x=splain_base_x_1, y=y_acc_1 - splain_base_y_1,mode='lines',name='Обычный Сплайн'))
fig.add_trace(go.Scatter(x=splain_base_x_2, y=y_acc_2 - splain_base_y_2,mode='lines',name='Обычный Сплайн'))
# fig.add_trace(go.Scatter(x=x_acc, y=y_acc,mode='lines',name='Заданная функция',marker=dict(size=8)))
fig.update_layout(title='Интерполяция сплайнами',xaxis_title='x',yaxis_title='y')
fig.show()









900
1900


In [11]:

# fig = go.Figure()
# fig.add_trace(go.Scatter(
#     x=splain_tension_x, 
#     y=np.array(splain_tension_y)-np.array(splain_base_y),
#     mode='lines',
#     name='Сплайн c натяжением'
# ))
# # fig.add_trace(go.Scatter(
# #     x=splain_base_x, 
# #     y=splain_base_y,
# #     mode='lines',
# #     name='Обычный Сплайн'
# # ))
# # fig.add_trace(go.Scatter(
# #     x=x_nodes, 
# #     y=y_nodes,
# #     mode='markers',
# #     name='Узлы',
# #     marker=dict(size=8)
# # ))
# fig.update_layout(
#     title='Ошибка обычного сплайна и сплайна с натяжением',
#     xaxis_title='x',
#     yaxis_title='y'
# )
# fig.show()